In [1]:
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset

DATASET_NAME = "bitext/Bitext-customer-support-llm-chatbot-training-dataset"

dataset = load_dataset(DATASET_NAME, split="train")
df = dataset.to_pandas()

df.head()

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


### Basic Dataset Information

In [2]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

Rows: 26872
Columns: 5

Columns:
['flags', 'instruction', 'category', 'intent', 'response']

Missing values:
flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64

Duplicate rows:
0


### Inspecting the 11 Categories

In [3]:
categories = sorted(df["category"].unique())

print(f"Number of categories: {len(categories)}")

for category in categories:
  print(category)

Number of categories: 11
ACCOUNT
CANCEL
CONTACT
DELIVERY
FEEDBACK
INVOICE
ORDER
PAYMENT
REFUND
SHIPPING
SUBSCRIPTION


### Inspecting all the 27 intents

In [4]:
intents = sorted(df['intent'].unique())

print(f"Number of intents: {len(intents)}")

for intent in intents:
  print(intent)

Number of intents: 27
cancel_order
change_order
change_shipping_address
check_cancellation_fee
check_invoice
check_payment_methods
check_refund_policy
complaint
contact_customer_service
contact_human_agent
create_account
delete_account
delivery_options
delivery_period
edit_account
get_invoice
get_refund
newsletter_subscription
payment_issue
place_order
recover_password
registration_problems
review
set_up_shipping_address
switch_account
track_order
track_refund


### Category Distribution

In [5]:
category_counts = df['category'].value_counts()

category_counts

category
ACCOUNT         5986
ORDER           3988
REFUND          2992
INVOICE         1999
CONTACT         1999
PAYMENT         1998
FEEDBACK        1997
DELIVERY        1994
SHIPPING        1970
SUBSCRIPTION     999
CANCEL           950
Name: count, dtype: int64

<small>Calculate percentage of category distribution

In [6]:
category_distribution = pd.DataFrame({
  "count": category_counts,
  "percentage": (category_counts / len(df) * 100).round(2)
})

category_distribution

,count,percentage
category,,
ACCOUNT,5986,22.28
ORDER,3988,14.84
REFUND,2992,11.13
INVOICE,1999,7.44
CONTACT,1999,7.44
PAYMENT,1998,7.44
FEEDBACK,1997,7.43
DELIVERY,1994,7.42
SHIPPING,1970,7.33


### Intent Distribution

In [7]:
intent_counts = df['intent'].value_counts()

intent_distribution = pd.DataFrame({
  "count": intent_counts,
  "percentage": (intent_counts / len(df) * 100).round(2)
})

intent_distribution

,count,percentage
intent,,
check_invoice,1000,3.72
complaint,1000,3.72
contact_customer_service,1000,3.72
edit_account,1000,3.72
switch_account,1000,3.72
check_payment_methods,999,3.72
contact_human_agent,999,3.72
delivery_period,999,3.72
get_invoice,999,3.72


### Verifying which category each intent belongs to

In [8]:
intent_category_mapping = (
  df[["intent", "category"]]
  .drop_duplicates()
  .sort_values(["category","intent"])
)

intent_category_mapping

,intent,category
9913,create_account,ACCOUNT
10910,delete_account,ACCOUNT
13899,edit_account,ACCOUNT
19891,recover_password,ACCOUNT
20886,registration_problems,ACCOUNT
23879,switch_account,ACCOUNT
2968,check_cancellation_fee,CANCEL
7914,contact_customer_service,CONTACT
8914,contact_human_agent,CONTACT
11905,delivery_options,DELIVERY


In [9]:
intent_category_counts = (
  df.groupby("intent")["category"]
  .nunique()
  .sort_values(ascending=False)
)

intent_category_counts

intent
cancel_order                1
change_order                1
change_shipping_address     1
check_cancellation_fee      1
check_invoice               1
check_payment_methods       1
check_refund_policy         1
complaint                   1
contact_customer_service    1
contact_human_agent         1
create_account              1
delete_account              1
delivery_options            1
delivery_period             1
edit_account                1
get_invoice                 1
get_refund                  1
newsletter_subscription     1
payment_issue               1
place_order                 1
recover_password            1
registration_problems       1
review                      1
set_up_shipping_address     1
switch_account              1
track_order                 1
track_refund                1
Name: category, dtype: int64

### Check if any intent belongs to more than 1 category

In [10]:
inconsistent_intents = intent_category_counts[
    intent_category_counts > 1
]

if len(inconsistent_intents) == 0:
    print("All intents map to exactly one category.")
else:
    print("Warning: Some intents map to multiple categories.")
    print(inconsistent_intents)

All intents map to exactly one category.
